In [1]:
!pip install -q sentence-transformers faiss-gpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.5/85.5 MB 20.5 MB/s eta 0:00:00:00:0100:01


In [2]:
import pandas as pd
import pickle
import sentence_transformers

In [3]:
import re

def keep_only_words(name):
    words = name.split()
    filtered_words = []
    for word in words:
        if not(any([c.isdigit() for c in word])):
            filtered_words.append(word.lower())
    return ' '.join(filtered_words)

def remove_english_words(text):
    return re.sub(r'\b\w*[a-zA-Z]\w*\b', '', text).strip()

def clean_text(text):
    text = re.sub(r"[^\w\s]", " ", text)  # Remove special characters
    return re.sub(r"\s+", " ", text).strip()  # Normalize spaces

def preprocess_names(names, preprocess_functions):

  for func in preprocess_functions:
    names = [func(name) for name in names]

  print('Names were preprocessed')
  return names

In [4]:
import faiss
import numpy as np

class FaissIVFIndex:
    def __init__(self, nlist=100, nprobe=10, use_gpu=True):
        """
        Initialize the Faiss IVF index.

        :param embeddings: numpy array of shape (num_vectors, embedding_dim)
        :param nlist: number of Voronoi cells (centroids) for IVF
        :param nprobe: number of cells to probe during search
        :param use_gpu: whether to use GPU for indexing and search
        """
        self.nlist = nlist
        self.nprobe = nprobe
        self.use_gpu = use_gpu

    def train(self,  embeddings):
        self.embeddings = embeddings
        self.num_vectors, self.embedding_dim = embeddings.shape
        # Normalize embeddings for cosine similarity
        faiss.normalize_L2(self.embeddings)

        # Create the IVF index
        self.index = faiss.IndexIVFFlat(
            faiss.IndexFlatIP(self.embedding_dim),  # Inner product index for cosine similarity
            self.embedding_dim,
            self.nlist,
            faiss.METRIC_INNER_PRODUCT
        )

        if self.use_gpu:
            # Move the index to GPU
            res = faiss.StandardGpuResources()
            print('GPU resources for Faiss index:\n', res)
            self.index = faiss.index_cpu_to_gpu(res, 0, self.index)

        # Train the index
        self.index.train(self.embeddings)

        # Add embeddings to the index
        self.index.add(self.embeddings)
        cpu_index = faiss.index_gpu_to_cpu(self.index)
        faiss.write_index(cpu_index, "index_file.index")
        print('Faiss index was saved locally.')

def search_in_index(index_path, query_embeddings, nprobe, k=5):
      """
      Search for similar embeddings.

      :param query_embeddings: numpy array of shape (num_queries, embedding_dim)
      :param k: number of nearest neighbors to return
      :return: distances, indices
      """
      # Normalize query embeddings for cosine similarity
      cpu_index = faiss.read_index('index_file.index')
      res = faiss.StandardGpuResources()
      gpu_index = faiss.index_cpu_to_gpu(res, 0, cpu_index)
      faiss.normalize_L2(query_embeddings)

      # Set the number of probes
      gpu_index.nprobe = nprobe

      # Perform the search
      distances, indices = gpu_index.search(query_embeddings, k)

      return distances, indices

In [5]:
from tqdm import tqdm


def voting_prediction(neighbors_indexes, neighbors_distances, categories):
    predicted_labels = []
    for neighbor_indices in tqdm(neighbors_indexes):
      neighbor_labels = [categories[i] for i in neighbor_indices]
      # Use majority voting or other logic to determine the final label
      final_label = max(set(neighbor_labels), key=neighbor_labels.count)
      predicted_labels.append(final_label)
    return predicted_labels

In [6]:
from sentence_transformers import SentenceTransformer
import torch


class ItemCategoryPredictor:

    def __init__(self, k_neighbors, emb_model_name, local_path='local_emb_model'):
      self.k_neigbors = k_neighbors
      self.emb_model_name = emb_model_name
      self.local_path = local_path
      self.device = 'cuda' if torch.cuda.is_available() else 'cpu'
      print(f'Device available for embedder: {self.device}')
      model = SentenceTransformer(self.emb_model_name)
      model.save(local_path)
      print(f'Embedder model saved locally to {self.local_path}')

    def _get_embeddings(self, texts, local_path='local_emb_model'):
      emb_model = SentenceTransformer(local_path)
      print('Calculating embeddings...')
      embs = emb_model.encode(texts, device=self.device)
      return embs

    def fit(self, items_names, categories):
      #preprocess names
      self.categories = categories
      processed_names = preprocess_names(items_names, [keep_only_words, remove_english_words, clean_text])
      # get embeddings
      names_embeddings = self._get_embeddings(processed_names)
      # train and save Faiss index
      faiss_idx = FaissIVFIndex()
      faiss_idx.train(names_embeddings)

    def predict(self, items_names, nprobe, index_path='index_file.index'):
      #preprocess names
      processed_names = preprocess_names(items_names, [keep_only_words, remove_english_words, clean_text])
      names_embs = self._get_embeddings(processed_names)
      #retrive nearest neighbors and distances from trained index
      distances, indexes = search_in_index(index_path, names_embs, nprobe, k=5) # make in per test item with tqdm??
      #make prediction
      predictions = voting_prediction(indexes, distances, self.categories)
      return predictions

In [7]:
#on train time

predictor = ItemCategoryPredictor(k_neighbors=5, emb_model_name='deepvk/USER-bge-m3')
N_train = 700000
train = pd.read_parquet('/kaggle/input/labelcraftdataset/labeled_train.parquet').head(N_train)
predictor.fit(train['source_name'], train['cat_id'])

Device available for embedder: cuda


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/195 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/9.34k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/963 [00:00<?, ?B/s]

1_Pooling%2Fconfig.json:   0%|          | 0.00/297 [00:00<?, ?B/s]

Embedder model saved locally to local_emb_model
Names were preprocessed
Calculating embeddings...


Batches:   0%|          | 0/21875 [00:00<?, ?it/s]

GPU resources for Faiss index:
 <faiss.swigfaiss.StandardGpuResources; proxy of <Swig Object of type 'faiss::gpu::StandardGpuResources *' at 0x7b042d383150> >
Faiss index was saved locally.


In [8]:
with open('item_cat_predictor.pkl', 'wb') as f:
    pickle.dump(predictor, f)

In [9]:
# on test time


with open('item_cat_predictor.pkl', 'rb') as f:
  predictor = pickle.load(f)

N_test = 1000
test = pd.read_parquet('/kaggle/input/labelcraftdataset/unlabeled_train.parquet').head(N_test)

pred_categories = predictor.predict(test['source_name'], nprobe=3)

Names were preprocessed
Calculating embeddings...


Batches:   0%|          | 0/32 [00:00<?, ?it/s]

100%|██████████| 1000/1000 [00:00<00:00, 69362.88it/s]


In [14]:
test_df = pd.DataFrame()
cat_df = pd.read_csv('/kaggle/input/labelcraftdataset/category_tree.csv')
test_df['name'] = test['source_name']
test_df['cat_id'] = pred_categories

test_df.merge(cat_df, on='cat_id').sample(5)

,name,cat_id,parent_id,cat_name
478,"Кабель питания, черный",10421,1308.0,"Кабели, разъемы для ПК и электроники"
222,Аккумулятор для eMachines E730ZG 11.1V 5200mAh,1370,132.0,Аккумуляторные батареи
619,Беспроводная мышь 3DCONNEXION SpaceMouse черны...,10041,1015.0,Беспроводные мыши
342,"Лобзик Bosch 12V-70, без АКБ, без ЗУ [06015a1001]",1259,121.0,Компьютерные корпуса
662,Чехол MyPads для Tecno Spark 5 Air Blue (153164),1081,107.0,Аксессуары для наушников и гарнитур


In [15]:
%%writefile run.py

import faiss
import pickle
import re
from sentence_transformers import SentenceTransformer
import torch
import argparse


def keep_only_words(name):
    words = name.split()
    filtered_words = []
    for word in words:
        if not(any([c.isdigit() for c in word])):
            filtered_words.append(word.lower())
    return ' '.join(filtered_words)

def remove_english_words(text):
    return re.sub(r'\b\w*[a-zA-Z]\w*\b', '', text).strip()

def clean_text(text):
    text = re.sub(r"[^\w\s]", " ", text)  # Remove special characters
    return re.sub(r"\s+", " ", text).strip()  # Normalize spaces

def preprocess_names(names, preprocess_functions):

  for func in preprocess_functions:
    names = [func(name) for name in names]

  print('Names were preprocessed')
  return names

def search_in_index(index_path, query_embeddings, nprobe, k=5):
      """
      Search for similar embeddings.

      :param query_embeddings: numpy array of shape (num_queries, embedding_dim)
      :param k: number of nearest neighbors to return
      :return: distances, indices
      """
      # Normalize query embeddings for cosine similarity
      cpu_index = faiss.read_index('index_file.index')
      res = faiss.StandardGpuResources()
      gpu_index = faiss.index_cpu_to_gpu(res, 0, cpu_index)
      faiss.normalize_L2(query_embeddings)

      # Set the number of probes
      gpu_index.nprobe = nprobe

      # Perform the search
      distances, indices = gpu_index.search(query_embeddings, k)

      return distances, indices

def voting_prediction(neighbors_indexes, neighbors_distances, categories):
    predicted_labels = []
    for neighbor_indices in neighbors_indexes:
      neighbor_labels = [categories[i] for i in neighbor_indices]
      # Use majority voting or other logic to determine the final label
      final_label = max(set(neighbor_labels), key=neighbor_labels.count)
      predicted_labels.append(final_label)
    return predicted_labels


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument('--test_data_path', type=str, help='test data path')
    parser.add_argument('--output_path', type=str, help='output file')
    args = parser.parse_args()

    test_data = pd.read_parquet(args.test_data_path)

    with open("item_cat_predictor.pkl", "rb") as f:
        model = pickle.load(f)
    
    predictions = model.predict(test_data['source_name'], nprobe=3)
    test_data['predicted_cat'] = predictions
    test_data[['hash_id', 'predicted_cat']].to_csv(args.output_path, index=False)


if __name__ == "__main__":
    main()


Writing run.py


In [21]:
%%writefile metadata.json
{
    "image": "hitchiker10/ml-competition:v1.0",
    "entry_point": "python -u run.py"
}

Writing metadata.json


In [22]:
!mkdir solution
!mv /kaggle/working/*.pkl /kaggle/working/run.py /kaggle/working/metadata.json /kaggle/working/index_file.index solution
!cd /kaggle/working
!zip  -r solution.zip solution

mkdir: cannot create directory ‘solution’: File exists
mv: cannot stat '/kaggle/working/*.pkl': No such file or directory
mv: cannot stat '/kaggle/working/run.py': No such file or directory
mv: cannot stat '/kaggle/working/index_file.index': No such file or directory
updating: solution/ (stored 0%)
updating: solution/metadata.json (deflated 8%)
updating: solution/item_cat_predictor.pkl (deflated 67%)
updating: solution/run.py (deflated 58%)
updating: solution/index_file.index (deflated 30%)


In [23]:
!unzip -l solution.zip

Archive:  solution.zip
  Length      Date    Time    Name
---------  ---------- -----   ----
        0  2025-02-23 20:09   solution/
       90  2025-02-23 20:09   solution/metadata.json
  2800797  2025-02-23 18:19   solution/item_cat_predictor.pkl
     2570  2025-02-23 19:22   solution/run.py
2873210539  2025-02-23 18:19   solution/index_file.index
---------                     -------
2876013996                     5 files


In [24]:
from IPython.display import FileLink
FileLink(r'solution.zip')

/kaggle/working/solution.zip

In [27]:
!pip freeze -> requirements.txt